In [ ]:
import scanpy as sc
import muon as mu
import numpy as np
import pandas as pd
import mofax as mofa
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (12,10)

In [ ]:
he_id = 'V11L12-109_B1'
gene_sample_id = he_id + '_visium'
adata_gene = sc.read_h5ad(f'MSi_{gene_sample_id}_full_adata.h5ad')
adata_msi = sc.read_h5ad(f'{he_id}_plusplus_sub2_norm_MALDI_adata.h5ad')

In [ ]:
adata_msi

In [ ]:
adata_gene

In [ ]:
# '806.542'
plt.scatter(adata_msi.obsm['spatial'][:, 0], adata_msi.obsm['spatial'][:, 1], c = adata_msi[:, 0].X.toarray(), s = 0.02, vmax=7, vmin=-1)

In [ ]:
plt.scatter(adata_msi.obsm['spatial'][:, 0], adata_msi.obsm['spatial'][:, 1], c = adata_msi[:, 0].X.toarray(), s = 0.03, vmax=8, vmin=-2)

In [ ]:
adata_gene.X = adata_gene.layers['counts'].copy()
adata_msi.X = adata_msi.layers['counts'].copy()

In [ ]:
adata_msi_sub = adata_msi[:, :11].copy()
adata_msi_sub.var_names

In [ ]:
from mudata import MuData
mdata = MuData({
    "gene": adata_gene,
    "msi": adata_msi_sub
})

In [ ]:
mdata

In [ ]:
mu.tl.mofa(mdata, use_obs='union',
           n_factors=15, convergence_mode='medium',
           outfile="models/Mousebrain_FMP10_norm_Visium_union.hdf5")

In [ ]:
mdata.obsm['X_mofa'].shape

In [ ]:
sc.pp.neighbors(mdata, use_rep="X_mofa", metric='cosine', random_state=0)
sc.tl.umap(mdata, min_dist=0.0, random_state=0)

In [ ]:
sc.tl.leiden(mdata, resolution = 0.5, key_added = "leiden_0.5", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
sc.tl.leiden(mdata, resolution = 0.6, key_added = "leiden_0.6", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(mdata, resolution = 0.8, key_added = "leiden_0.8", random_state=0, flavor='igraph', n_iterations=2)
sc.tl.leiden(mdata, resolution = 0.4, key_added = "leiden_0.4", random_state=0, flavor='igraph', n_iterations=2)
#sc.tl.leiden(mdata, resolution = 1.0, key_added = "leiden_1.0", random_state=0, flavor='igraph', n_iterations=2)

In [ ]:
mdata

In [ ]:
sc.pl.umap(mdata['gene'], color='leiden_0.7', size=2.0)

In [ ]:
sc.pl.umap(mdata['msi'], color='leiden_0.4_sc', size=2.0)

In [ ]:
sc.pl.umap(mdata, color='leiden_0.4', size=2.0)
sc.pl.umap(mdata, color='leiden_0.6', size=2.0)
sc.pl.umap(mdata, color='leiden_0.8', size=2.0)

In [ ]:
mdata.uns['spatial'] = mdata['gene'].uns['spatial']
mdata.obsm['spatial'] = mdata['gene'].obsm['spatial']

In [ ]:
mdata

In [ ]:
mdata.uns['leiden_0.5_colors'] = ['#1f77b4',
                                 '#ff7f0e',
                                 '#279e68',
                                 '#d62728',
                                 '#aa40fc',
                                 '#8c564b',
                                 '#e377c2',
                                 '#b5bd61',
                                 '#17becf',
                                 '#aec7e8',
                                 '#ffbb78',
                                 '#98df8a',
                                 '#ff9896',
                                 '#c5b0d5',
                                 #'#c49c94'
                                  '#66FF00'
                                 ]

In [ ]:
sc.pl.spatial(mdata, color='leiden_0.5', groups='12', img_key=None, size=4.0)

In [ ]:
sc.pl.spatial(mdata, color='leiden_0.5', groups='8', img_key=None, size=4.0)

In [ ]:
sc.pl.spatial(mdata, color='leiden_0.5', img_key=None, size=4.0)

In [ ]:
sc.pl.spatial(mdata['msi'], color='leiden_0.3_sc', img_key=None, size=6.0)

In [ ]:
sc.pl.spatial(mdata['gene'], color='leiden_0.4', img_key=None, size=4.0)

In [ ]:
mdata['gene'].obs['leiden_0.5_joint'] = mdata.obs['leiden_0.5'].copy()
mdata['msi'].obs['leiden_0.5_joint'] = mdata.obs['leiden_0.5'].copy()

In [ ]:
mdata['gene'].X = mdata['gene'].layers['counts'].copy()
sc.pp.normalize_total(mdata['gene'], target_sum=1e4)
sc.pp.log1p(mdata['gene'])

In [ ]:
mdata

In [ ]:
mdata['gene'].write(f'{he_id}_RNA_norm_maldi_joint_cluster.h5ad')